In [3]:
import pandas as pd
import numpy as np
import time
import sys
import subprocess

from pathlib import Path
from collections import Counter
from gensim.models import Word2Vec, FastText

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
from pathlib import Path

ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "cleaned_tinystories.csv"
MODEL_DIR = ROOT / "models"

print("Project Root:", ROOT)
print("Dataset Path:", DATA_PATH)
print("Model Directory:", MODEL_DIR)

Project Root: c:\Users\hp\Desktop\nlp-project-tinyGpt
Dataset Path: c:\Users\hp\Desktop\nlp-project-tinyGpt\data\cleaned_tinystories.csv
Model Directory: c:\Users\hp\Desktop\nlp-project-tinyGpt\models


In [5]:

result = subprocess.run(
    [sys.executable, str(ROOT / "data" / "download_tinystories.py")],
    cwd=str(ROOT),
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Dataset download failed.")

print("Raw dataset downloaded successfully!")


scanned 20,000 | Dialogue=5000 MoralValue=646 Conflict=597
scanned 60,000 | Dialogue=5000 MoralValue=1936 Conflict=1656
scanned 140,000 | Dialogue=5000 MoralValue=4643 Conflict=3911

Scanned 176,551 stories. Tag frequencies:
  <none>          77,462
  Dialogue        70,206
  Twist           20,008
  MoralValue      13,619
  BadEnding       11,835
  Foreshadowing   11,821
  Conflict        11,740

Saved 15,000 stories -> C:\Users\hp\Desktop\nlp-project-tinyGpt\data\raw_tinystories.csv
narrative_type
Conflict      5000
Dialogue      5000
MoralValue    5000

Raw dataset downloaded successfully!


In [7]:

result = subprocess.run(
    [sys.executable, str(ROOT / "src" / "clean_pipeline.py")],
    cwd=str(ROOT),
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Data cleaning failed.")

print("Dataset cleaning completed!")


Loaded 15,000 raw stories
Label check: 0 mismatches
Lemmatized in 291s
Saved 15,000 stories -> C:\Users\hp\Desktop\nlp-project-tinyGpt\data\cleaned_tinystories.csv

Dataset cleaning completed!


In [8]:

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)
print("\nAvailable Columns:")
print(df.columns.tolist())

df.head()


Dataset Shape: (15000, 5)

Available Columns:
['id', 'label', 'story_clean', 'text_processed', 'n_words']


,id,label,story_clean,text_processed,n_words
0,0,Conflict,"Once upon a time, there was a big whale swimmi...",once upon a time there be a big whale swimming...,152
1,1,Conflict,"Once upon a time, there was a bug named Benny....",once upon a time there be a bug name benny ben...,148
2,2,Dialogue,"Once upon a time, in a small house, there live...",once upon a time in a small house there live a...,152
3,3,Conflict,"Once upon a time, there was a little girl name...",once upon a time there be a little girl name l...,133
4,4,Dialogue,"Once upon a time, in a small country, there li...",once upon a time in a small country there live...,169


In [9]:

sentences = [
    text.split()
    for text in df["text_processed"].fillna("")
    if text.strip()
]

total_tokens = sum(len(sentence) for sentence in sentences)

print("Total Stories:", len(sentences))
print("Total Tokens:", f"{total_tokens:,}")
print("First Sentence:", sentences[0][:20])


Total Stories: 15000
Total Tokens: 2,602,648
First Sentence: ['once', 'upon', 'a', 'time', 'there', 'be', 'a', 'big', 'whale', 'swimming', 'in', 'the', 'ocean', 'the', 'whale', 'be', 'very', 'happy', 'because', 'the']


In [10]:

PARAMS = {
    "vector_size": 100,
    "window": 5,
    "min_count": 5,
    "epochs": 10,
    "workers": 4,
    "seed": 42
}

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Hyperparameters:")
print(PARAMS)


Hyperparameters:
{'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 10, 'workers': 4, 'seed': 42}


In [11]:

start_time = time.time()

cbow = Word2Vec(
    sentences=sentences,
    sg=0,
    **PARAMS
)

cbow_time = time.time() - start_time

cbow.save(str(MODEL_DIR / "cbow.model"))

print("CBOW Training Completed!")
print("Vocabulary Size:", len(cbow.wv))
print("Training Time:", round(cbow_time, 2), "seconds")


CBOW Training Completed!
Vocabulary Size: 3838
Training Time: 11.8 seconds


In [12]:

start_time = time.time()

skipgram = Word2Vec(
    sentences=sentences,
    sg=1,
    **PARAMS
)

skipgram_time = time.time() - start_time

skipgram.save(str(MODEL_DIR / "skipgram.model"))

print("Skip-gram Training Completed!")
print("Vocabulary Size:", len(skipgram.wv))
print("Training Time:", round(skipgram_time, 2), "seconds")


Skip-gram Training Completed!
Vocabulary Size: 3838
Training Time: 58.61 seconds


In [13]:

comparison = pd.DataFrame([
    {
        "Model": "CBOW",
        "Architecture": "Context to Target",
        "Vocabulary": len(cbow.wv),
        "Training Time": round(cbow_time, 2)
    },
    {
        "Model": "Skip-gram",
        "Architecture": "Target to Context",
        "Vocabulary": len(skipgram.wv),
        "Training Time": round(skipgram_time, 2)
    }
])

comparison


,Model,Architecture,Vocabulary,Training Time
0,CBOW,Context to Target,3838,11.80
1,Skip-gram,Target to Context,3838,58.61


In [14]:

test_words = ["king", "queen", "story", "happy", "boy"]

for word in test_words:
    print(f"\nWord: {word}")

    if word in cbow.wv:
        print("CBOW:", cbow.wv.most_similar(word, topn=5))
    else:
        print("CBOW: Word not in vocabulary")

    if word in skipgram.wv:
        print("Skip-gram:", skipgram.wv.most_similar(word, topn=5))
    else:
        print("Skip-gram: Word not in vocabulary")



Word: king
CBOW: [('queen', 0.8051196932792664), ('prince', 0.722051203250885), ('kingdom', 0.6669509410858154), ('wizard', 0.6312763094902039), ('person', 0.601806640625)]
Skip-gram: [('prince', 0.7050948739051819), ('queen', 0.7027262449264526), ('kingdom', 0.6898404955863953), ('knight', 0.5813165307044983), ('princess', 0.5726487040519714)]

Word: queen
CBOW: [('king', 0.8051195740699768), ('prince', 0.6531203389167786), ('princess', 0.6204690337181091), ('wizard', 0.6158338189125061), ('woman', 0.5913414359092712)]
Skip-gram: [('king', 0.7027262449264526), ('crown', 0.6480942368507385), ('prince', 0.6349228620529175), ('princess', 0.6149693131446838), ('wizard', 0.5711645483970642)]

Word: story
CBOW: [('value', 0.5487562417984009), ('aware', 0.44689124822616577), ('important', 0.44457119703292847), ('learn', 0.4361921548843384), ('tale', 0.4280829429626465)]
Skip-gram: [('moral', 0.8357543349266052), ('considerate', 0.5685685276985168), ('suggestion', 0.5664498209953308), ('stor

In [15]:

word_counts = Counter(
    word
    for sentence in sentences
    for word in sentence
)

rare_words = [
    word for word, count in word_counts.items()
    if count == 2 and len(word) > 6
][:10]

print("Rare Words:")
print(rare_words)


Rare Words:
['waffles', 'dishearten', 'brushing', 'pianist', 'background', 'mesmerised', 'fragrance', 'wetness', 'sweetly', 'entirely']


In [16]:

start_time = time.time()

ft = FastText(
    sentences=sentences,
    sg=1,
    min_n=3,
    max_n=6,
    bucket=200_000,
    **PARAMS
)

fasttext_time = time.time() - start_time

ft.save(str(MODEL_DIR / "fasttext.model"))

print("fastText Training Completed!")
print("Vocabulary Size:", len(ft.wv))
print("Training Time:", round(fasttext_time, 2), "seconds")


fastText Training Completed!
Vocabulary Size: 3838
Training Time: 101.49 seconds


In [17]:

test_words = ["pianist", "dinosaurr", "happpy", "puppys"]

for word in test_words:
    print(f"\nTesting Word: {word}")

    if word in cbow.wv:
        print("Word2Vec:", cbow.wv.most_similar(word, topn=3))
    else:
        print("Word2Vec: No vector available")

    print(
        "fastText:",
        ft.wv.most_similar(word, topn=3)
    )



Testing Word: pianist
Word2Vec: No vector available
fastText: [('piano', 0.7585887908935547), ('spanish', 0.6960936784744263), ('saxophone', 0.6651321649551392)]

Testing Word: dinosaurr
Word2Vec: No vector available
fastText: [('dinosaur', 0.9860413670539856), ('unicorn', 0.6110374331474304), ('shiloh', 0.5719265937805176)]

Testing Word: happpy
Word2Vec: No vector available
fastText: [('happy', 0.7812333106994629), ('joyful', 0.6578461527824402), ('goodness', 0.6339051127433777)]

Testing Word: puppys
Word2Vec: No vector available
fastText: [('puppy', 0.972381591796875), ('puppet', 0.7422475218772888), ('pup', 0.7197155356407166)]


In [18]:

final_comparison = pd.DataFrame([
    {
        "Model": "CBOW",
        "Vocabulary": len(cbow.wv),
        "Training Time": round(cbow_time, 2)
    },
    {
        "Model": "Skip-gram",
        "Vocabulary": len(skipgram.wv),
        "Training Time": round(skipgram_time, 2)
    },
    {
        "Model": "fastText",
        "Vocabulary": len(ft.wv),
        "Training Time": round(fasttext_time, 2)
    }
])

final_comparison


,Model,Vocabulary,Training Time
0,CBOW,3838,11.80
1,Skip-gram,3838,58.61
2,fastText,3838,101.49


In [19]:

print("Saved Model Files:")

for file in MODEL_DIR.glob("*.model"):
    print(file.name, "-", round(file.stat().st_size / (1024 * 1024), 2), "MB")


Saved Model Files:
cbow.model - 3.04 MB
fasttext.model - 3.04 MB
skipgram.model - 3.04 MB


# Conclusion

In this assignment, three word embedding models were trained using the TinyStories dataset.

1. **CBOW:** Predicts a target word using its surrounding context. It is generally faster to train.
2. **Skip-gram:** Predicts surrounding context words using a target word. It can perform better on less frequent words.
3. **fastText:** Uses character-level n-grams to generate word representations. It can produce vectors for words that are not present in the training vocabulary.

All three models use 100-dimensional word vectors, a context window of 5, minimum word frequency of 5, and 10 training epochs.

The models were saved for future use and comparison.